# Digital Clone Bot - LoRA QLoRA Fine-Tuning Notebook
This notebook contains the complete pipeline to fine-tune a 7B LLM (like Qwen2.5-7B-Instruct or Mistral-7B-Instruct) on Google Colab's free T4 GPU tier. It uses **QLoRA** (4-bit quantization + Low Rank Adapters) to fit the model within Colab's 15GB VRAM limit.

## Training Overview
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct` (or `Qwen/Qwen2.5-3B-Instruct` / `mistralai/Mistral-7B-Instruct-v0.3`)
- **Dataset**: `final_train_dataset.jsonl` (Alpaca format: Instruction, Input, Output)
- **Method**: QLoRA with Hugging Face `peft`, `bitsandbytes`, and `trl`'s `SFTTrainer`.

### Step 1: Install Dependencies
Run this cell to install the necessary packages for quantization and fine-tuning.

In [ ]:
!pip install -q -U trl transformers peft accelerate bitsandbytes datasets

### Step 2: Import Libraries and Setup Config

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    logging
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Config Variables
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # or "Qwen/Qwen2.5-3B-Instruct" for faster training
DATASET_PATH = "final_train_dataset.jsonl"  # Upload your final_train_dataset.jsonl file here
OUTPUT_DIR = "./results_lora_clone"
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 3

### Step 3: Load the Dataset
We load the Alpaca-formatted training file. Upload it to the local Colab folder (click the folder icon on the left panel).

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded dataset with {len(dataset)} entries.")
print("Sample entry:")
print(dataset[0])

### Step 4: Load Base Model with 4-bit Quantization
We configure `BitsAndBytesConfig` to load the model parameters in 4-bit NF4 format with float16 compute capabilities.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading 4-bit base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for QLoRA training (enables gradient checkpointing)
base_model = prepare_model_for_kbit_training(base_model)
print("Model loaded and prepared for kbit training.")

### Step 5: Configure LoRA Adapters
We configure PEFT LoRA. We target attention and linear layer matrices (`q_proj`, `v_proj`, etc.) to get high quality stylistic updates.

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

### Step 6: Define formatting function
Formulate prompts in ChatML format or simple standard layout:

In [ ]:
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        inst = example['instruction'][i]
        inp = example['input'][i]
        out = example['output'][i]
        
        # Format in standard model templates
        text = f"<system>\n{inst}\n</system>\n<user>\n{inp}\n</user>\n<assistant>\n{out}"
        output_texts.append(text)
    return output_texts

### Step 7: Define Training Arguments & Run SFTTrainer

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    max_seq_length=512,
    tokenizer=tokenizer,
    formatting_func=formatting_prompts_func,
    args=training_args,
)

# Start training
trainer.train()

### Step 8: Save and Push Adapters
After training, save the LoRA weights locally. You can download these weights and place them in the `src/inference/` folder of your project.

In [ ]:
# Save locally
trainer.model.save_pretrained("./best_lora_adapter")
tokenizer.save_pretrained("./best_lora_adapter")
print("Adapter weights saved to ./best_lora_adapter")

# Optional: Push to Hugging Face Hub (requires notebook_login)
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.model.push_to_hub("abhinav/digital-clone-lora-adapter")